In [1]:
!pip install onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.4 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import json, glob, math, os
import numpy as np

class OrbitNet(nn.Module):
    def __init__(self):
        super(OrbitNet, self).__init__()
        self.fc1 = nn.Linear(8, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.sigmoid(self.fc3(x))

def extract_training_data(dataset_path, max_files=200):
    files = glob.glob(os.path.join(dataset_path, "**/*.json"), recursive=True)
    X_data, Y_data = [], []
    
    for filepath in files[:max_files]:
        try:
            with open(filepath, 'r') as f: match = json.load(f)
            steps = match.get("steps",[])
            if not steps: continue
            
            winner_id = np.argmax([agent.get("reward", -1) for agent in steps[-1]])
            
            for step_idx, step in enumerate(steps[:-1]):
                winner_obs = step[0].get("observation", {})
                planets = winner_obs.get("planets",[])
                winner_action = step[winner_id].get("action",[])
                is_4p = 1.0 if len(set([p[1] for p in planets if p[1] != -1])) > 2 else 0.0
                
                for action in winner_action:
                    src_id, angle, ships = action[0], action[1], action[2]
                    src_p = next((p for p in planets if p[0] == src_id), None)
                    if not src_p: continue
                    
                    best_tgt = None
                    best_diff = float('inf')
                    for p in planets:
                        if p[0] == src_id: continue
                        tgt_ang = math.atan2(p[3] - src_p[3], p[2] - src_p[2])
                        ang_diff = abs((angle - tgt_ang + math.pi) % (2 * math.pi) - math.pi)
                        if ang_diff < best_diff: best_diff, best_tgt = ang_diff, p
                            
                    if best_tgt and best_diff < 0.2:
                        X_data.append(build_features(src_p, best_tgt, step_idx, is_4p))
                        Y_data.append([1.0]) # Good Move
                        
                        bad_tgt = planets[np.random.randint(len(planets))]
                        if bad_tgt[0] not in (src_id, best_tgt[0]):
                            X_data.append(build_features(src_p, bad_tgt, step_idx, is_4p))
                            Y_data.append([0.0]) # Bad Move
        except Exception:
            continue
    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data, dtype=torch.float32)

def build_features(src, tgt, step_idx, is_4p):
    dist = math.hypot(tgt[2] - src[2], tgt[3] - src[3])
    # Approximate ETA for training context
    eta = dist / (1.0 + 5.0 * (min(1.0, math.log(max(1, src[5])) / math.log(1000.0)) ** 1.5))
    
    # Mathematical Economic Features
    net_present_value = (tgt[6] * max(0, 500 - step_idx - eta)) / 1000.0
    capital_risk = tgt[5] / max(1.0, float(src[5]))
    time_discount = eta / 50.0
    
    in_quad = 1.0 if (1 if src[2]>50 else -1) == (1 if tgt[2]>50 else -1) and (1 if src[3]>50 else -1) == (1 if tgt[3]>50 else -1) else 0.0

    return[
        net_present_value,                  # 0: Expected ROI (Time-adjusted)
        capital_risk,                       # 1: Capital Depletion Risk
        time_discount,                      # 2: Time spent in transit
        tgt[6] / 5.0,                       # 3: Raw Production Yield
        1.0 if tgt[1] != -1 else 0.5,       # 4: Ownership Type
        in_quad,                            # 5: Geographic Proximity
        step_idx / 500.0,                   # 6: Game Phase
        is_4p                               # 7: 4-Player Match
    ]

def train_and_export():
    dataset_path = "/kaggle/input/datasets/bovard/orbit-wars-top10-episodes-2026-05-04/episodes/episodes"
    X_train, Y_train = extract_training_data(dataset_path, max_files=200)
    
    model = OrbitNet()
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.MSELoss()
    
    print("Training Model...")
    for epoch in range(100):
        optimizer.zero_grad()
        loss = criterion(model(X_train), Y_train)
        loss.backward()
        optimizer.step()
        
    model.eval() 
    torch.onnx.export(model, torch.randn(1, 8), "/kaggle/working/orbit_model.onnx", 
                      export_params=True, opset_version=18, input_names=['input'], output_names=['output'])
    print("Saved to /kaggle/working/orbit_model.onnx")

if __name__ == "__main__":
    train_and_export()

Training Model...
[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Saved to /kaggle/working/orbit_model.onnx


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [3]:
%%writefile main.py
import math
import numpy as np
import os
from collections import namedtuple

try: import onnxruntime as ort
except ImportError: ort = None

# ============================================================
# Engine Configuration
# ============================================================
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 0.5
OVERCOMMIT_RATIO = 1.1
MIN_NN_SCORE = 0.60 # Trust the model: must be >60% confident

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])

onnx_session = None
if ort:
    for path in["/kaggle/input/orbit-model-onnx/orbit_model.onnx", "orbit_model.onnx", "/kaggle_simulations/agent/orbit_model.onnx"]:
        if os.path.exists(path):
            try: onnx_session = ort.InferenceSession(path); break
            except: pass

# ============================================================
# Core Physics Engine (Autopilot)
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx, dy = x2 - x1, y2 - y1
    l2 = dx*dx + dy*dy
    if l2 == 0: return math.hypot(px - x1, py - y1)
    t = max(0.0, min(1.0, ((px - x1)*dx + (py - y1)*dy) / l2))
    return math.hypot(px - (x1 + t*dx), py - (y1 + t*dy))

def get_target_pos(tgt, turns, ang_vel, initial_planets, comets, comet_ids):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids",[]):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): return c["paths"][idx][f_idx]
        return None
    init = initial_planets.get(tgt.id)
    if not init: return (tgt.x, tgt.y)
    r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
    if r + init.radius >= 50.0: return (tgt.x, tgt.y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, initial_planets, comets, comet_ids):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    for _ in range(5): # Iterative solver
        flight_dist = max(0.0, math.hypot(tx - src.x, ty - src.y) - src.radius - 0.1 - tgt.radius)
        eta = flight_dist / speed
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos: return None, 999
        tx, ty = pos

    angle = math.atan2(ty - src.y, tx - src.x)
    sx = src.x + math.cos(angle) * (src.radius + 0.1)
    sy = src.y + math.sin(angle) * (src.radius + 0.1)
    ex = sx + math.cos(angle) * (eta * speed)
    ey = sy + math.sin(angle) * (eta * speed)
    
    if point_to_segment_dist(CENTER_X, CENTER_Y, sx, sy, ex, ey) <= SUN_R + SUN_SAFETY: return None, 999
    return angle, int(math.ceil(eta))

# ============================================================
# Execution Loop
# ============================================================
def agent(obs, config=None):
    is_dict = isinstance(obs, dict)
    player = obs.get("player", 0) if is_dict else getattr(obs, "player", 0)
    step = obs.get("step", 0) if is_dict else getattr(obs, "step", 0)
    
    planets =[Planet(*p) for p in (obs.get("planets",[]) if is_dict else getattr(obs, "planets",[]))]
    ang_vel = obs.get("angular_velocity", 0.0) if is_dict else getattr(obs, "angular_velocity", 0.0)
    initial_planets = {Planet(*p).id: Planet(*p) for p in (obs.get("initial_planets",[]) if is_dict else getattr(obs, "initial_planets",[]))}
    comets = obs.get("comets",[]) if is_dict else getattr(obs, "comets",[])
    comet_ids = set(obs.get("comet_planet_ids",[]) if is_dict else getattr(obs, "comet_planet_ids",[]))
    
    my_planets =[p for p in planets if p.owner == player]
    if not my_planets or not onnx_session: return[]
    
    is_4p = 1.0 if len(set([p.owner for p in planets if p.owner != -1])) > 2 else 0.0
    budget = {p.id: p.ships for p in my_planets}
    nn_evals, targeted =[], set()

    # 1. Ask ONNX to evaluate every possible move
    for src in my_planets:
        avail = budget[src.id]
        if avail < 10: continue

        for tgt in planets:
            if tgt.owner == player: continue
            
            # Autopilot: Physics Simulation
            rough_eta = math.hypot(tgt.x - src.x, tgt.y - src.y) / fleet_speed(avail)
            base_cost = tgt.ships + (tgt.production * rough_eta if tgt.owner != -1 else 0) + 1
            cost = int(math.ceil(base_cost * OVERCOMMIT_RATIO))
            
            if cost >= avail or cost < 1: continue
            
            angle, exact_eta = plan_flight(src, tgt, cost, ang_vel, initial_planets, comets, comet_ids)
            if angle is None or exact_eta > 45: continue 

            # CEO: Economic Valuation Vector
            in_quad = 1.0 if (1 if src.x>50 else -1) == (1 if tgt.x>50 else -1) and (1 if src.y>50 else -1) == (1 if tgt.y>50 else -1) else 0.0
            
            feature_vector = np.array([[
                (tgt.production * max(0, 500 - step - exact_eta)) / 1000.0, # Net Present Value
                cost / max(1.0, float(src.ships)),                          # Capital Risk
                exact_eta / 50.0,                                           # Time Discount
                tgt.production / 5.0,                                       # Raw Yield
                1.0 if tgt.owner != -1 else 0.5,                            # Ownership Type
                in_quad,                                                    # Geographic Priority
                step / 500.0,                                               # Game Phase
                is_4p                                                       # 4P Match
            ]], dtype=np.float32)

            score = onnx_session.run(None, {'input': feature_vector})[0][0][0]
            if score >= MIN_NN_SCORE: nn_evals.append((score, src.id, tgt.id, cost, angle))

    # 2. Execute the highest-scoring economic moves
    moves =[]
    nn_evals.sort(key=lambda x: x[0], reverse=True)
    
    for score, src_id, tgt_id, cost, angle in nn_evals:
        if budget[src_id] >= cost and tgt_id not in targeted:
            moves.append([src_id, angle, cost])
            budget[src_id] -= cost
            targeted.add(tgt_id)

    return moves

Writing main.py
